# MediaPipe + Gradio + MySQL(SQLAlchemy + PyMySQL) 얼굴 등록/인증 예제

이 노트북은 다음 흐름으로 동작합니다.

1. 웹캠으로 얼굴 등록
2. MediaPipe FaceMesh로 얼굴 랜드마크 벡터 추출
3. MySQL DB 저장
4. 인증 시도
5. DB 저장 데이터와 비교
6. 인증 성공 / 실패 판정

## 실행 순서
위에서 아래로 셀을 순서대로 실행하세요.

## 환경
- Jupyter Notebook / JupyterLab
- Windows
- MySQL
- Gradio는 브라우저에서 실행

In [ ]:
# !pip install sqlalchemy pymysql mediapipe gradio opencv-python numpy

In [7]:
# !pip install ipywidgets

In [1]:
import cv2
import time
import numpy as np
import gradio as gr
import mediapipe as mp

from sqlalchemy import create_engine, text

In [2]:
# =========================
# MySQL 접속 정보
# =========================
DB_HOST = "127.0.0.1"
DB_PORT = 3306
DB_USER = "root"
DB_PASSWORD = "1234"
DB_NAME = "face_auth_db"

# DB URL
DB_URL_SERVER = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/?charset=utf8mb4"
DB_URL_DB = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4"

# Engine
server_engine = create_engine(
    DB_URL_SERVER,
    pool_pre_ping=True,
    future=True
)

engine = create_engine(
    DB_URL_DB,
    pool_pre_ping=True,
    future=True
)

In [3]:
import sys
print(sys.executable)

C:\Users\Admin\miniforge3\envs\meddiapipe\python.exe


In [4]:
# MediaPipe 설정
mp_face_mesh = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils
mp_styles = mp.solutions.drawing_styles

In [5]:
def init_database():
    # 1) DB 생성
    with server_engine.begin() as conn:
        conn.execute(text(
            f"CREATE DATABASE IF NOT EXISTS {DB_NAME} "
            f"CHARACTER SET utf8mb4 COLLATE utf8mb4_general_ci"
        ))

    # 2) 테이블 생성
    with engine.begin() as conn:
        conn.execute(text("""
            CREATE TABLE IF NOT EXISTS face_users (
                id INT AUTO_INCREMENT PRIMARY KEY,
                user_name VARCHAR(100) NOT NULL UNIQUE,
                face_vector LONGBLOB NOT NULL,
                vector_dim INT NOT NULL,
                created_at DATETIME NOT NULL,
                updated_at DATETIME NOT NULL
            )
        """))

In [6]:
def save_face_to_db(user_name: str, face_vector: np.ndarray):
    now_str = time.strftime("%Y-%m-%d %H:%M:%S")
    vec = face_vector.astype(np.float32)
    vec_bytes = vec.tobytes()
    vec_dim = int(vec.shape[0])

    with engine.begin() as conn:
        row = conn.execute(
            text("SELECT id FROM face_users WHERE user_name = :user_name"),
            {"user_name": user_name}
        ).fetchone()

        if row is None:
            conn.execute(text("""
                INSERT INTO face_users (
                    user_name, face_vector, vector_dim, created_at, updated_at
                )
                VALUES (
                    :user_name, :face_vector, :vector_dim, :created_at, :updated_at
                )
            """), {
                "user_name": user_name,
                "face_vector": vec_bytes,
                "vector_dim": vec_dim,
                "created_at": now_str,
                "updated_at": now_str
            })
        else:
            conn.execute(text("""
                UPDATE face_users
                SET face_vector = :face_vector,
                    vector_dim = :vector_dim,
                    updated_at = :updated_at
                WHERE user_name = :user_name
            """), {
                "face_vector": vec_bytes,
                "vector_dim": vec_dim,
                "updated_at": now_str,
                "user_name": user_name
            })

def load_all_faces_from_db():
    users = []

    with engine.begin() as conn:
        rows = conn.execute(text("""
            SELECT id, user_name, face_vector, vector_dim, created_at, updated_at
            FROM face_users
            ORDER BY id ASC
        """)).fetchall()

    for row in rows:
        vec = np.frombuffer(row.face_vector, dtype=np.float32)
        if len(vec) != row.vector_dim:
            continue

        users.append({
            "id": row.id,
            "user_name": row.user_name,
            "face_vector": vec,
            "vector_dim": row.vector_dim,
            "created_at": str(row.created_at),
            "updated_at": str(row.updated_at),
        })

    return users

def delete_user_from_db(user_name: str):
    with engine.begin() as conn:
        result = conn.execute(
            text("DELETE FROM face_users WHERE user_name = :user_name"),
            {"user_name": user_name}
        )
        return result.rowcount

def get_registered_users_text():
    users = load_all_faces_from_db()
    if not users:
        return "등록된 사용자가 없습니다."

    lines = []
    for u in users:
        lines.append(
            f"[id={u['id']}] {u['user_name']} | created_at={u['created_at']} | updated_at={u['updated_at']}"
        )
    return "\n".join(lines)

def safe_get_registered_users_text():
    try:
        return get_registered_users_text()
    except Exception as e:
        return f"사용자 목록 조회 실패: {e}"

In [7]:
def bgr_from_rgb(image: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

def rgb_from_bgr(image: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

def draw_text(img_bgr, text_value, y=30, color=(0, 255, 0)):
    out = img_bgr.copy()
    cv2.putText(
        out,
        text_value,
        (10, y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        color,
        2,
        cv2.LINE_AA
    )
    return out

def face_landmarks_to_vector(face_landmarks):
    pts = np.array(
        [[lm.x, lm.y, lm.z] for lm in face_landmarks.landmark],
        dtype=np.float32
    )

    center = pts.mean(axis=0, keepdims=True)
    pts = pts - center

    scale = np.linalg.norm(pts, axis=1).mean()
    if scale < 1e-8:
        return None

    pts = pts / scale
    return pts.flatten()

def detect_face_vector_and_annotated(image_rgb: np.ndarray):
    if image_rgb is None:
        return None, None, "이미지가 없습니다."

    image_bgr = bgr_from_rgb(image_rgb)

    with mp_face_mesh.FaceMesh(
        static_image_mode=True,
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.5
    ) as face_mesh:
        results = face_mesh.process(image_rgb)

    annotated = image_bgr.copy()

    if not results.multi_face_landmarks:
        annotated = draw_text(annotated, "Face not detected", y=30, color=(0, 0, 255))
        return None, rgb_from_bgr(annotated), "얼굴을 찾지 못했습니다."

    face_landmarks = results.multi_face_landmarks[0]

    mp_drawing.draw_landmarks(
        image=annotated,
        landmark_list=face_landmarks,
        connections=mp_face_mesh.FACEMESH_TESSELATION,
        landmark_drawing_spec=None,
        connection_drawing_spec=mp_styles.get_default_face_mesh_tesselation_style()
    )

    vec = face_landmarks_to_vector(face_landmarks)
    if vec is None:
        annotated = draw_text(annotated, "Vectorization failed", y=30, color=(0, 0, 255))
        return None, rgb_from_bgr(annotated), "특징 벡터 생성 실패"

    annotated = draw_text(annotated, "Face detected", y=30, color=(0, 255, 0))
    return vec, rgb_from_bgr(annotated), "얼굴 검출 성공"

In [8]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom < 1e-8:
        return 0.0
    return float(np.dot(a, b) / denom)

def euclidean_distance(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.linalg.norm(a - b))

def enroll_face(image, user_name):
    user_name = (user_name or "").strip()

    if not user_name:
        return None, "사용자 이름을 입력하세요.", safe_get_registered_users_text()

    if image is None:
        return None, "웹캠으로 얼굴을 촬영하세요.", safe_get_registered_users_text()

    vec, annotated, msg = detect_face_vector_and_annotated(image)
    if vec is None:
        return annotated, f"등록 실패: {msg}", safe_get_registered_users_text()

    try:
        save_face_to_db(user_name, vec)
        return annotated, f"등록 완료: {user_name}", safe_get_registered_users_text()
    except Exception as e:
        return annotated, f"DB 저장 실패: {e}", safe_get_registered_users_text()

def verify_face(image, threshold, cosine_threshold):
    if image is None:
        return None, "웹캠으로 얼굴을 촬영하세요."

    vec, annotated, msg = detect_face_vector_and_annotated(image)
    if vec is None:
        return annotated, f"인증 실패: {msg}"

    users = load_all_faces_from_db()
    if not users:
        return annotated, "DB에 등록된 사용자가 없습니다."

    best_user = None
    best_dist = 1e9
    best_cos = -1.0

    for u in users:
        db_vec = u["face_vector"]
        if db_vec.shape != vec.shape:
            continue

        dist = euclidean_distance(db_vec, vec)
        cos = cosine_similarity(db_vec, vec)

        if dist < best_dist:
            best_dist = dist
            best_cos = cos
            best_user = u

    if best_user is None:
        return annotated, "비교 가능한 등록 데이터가 없습니다."

    is_match = (best_dist <= threshold) or (best_cos >= cosine_threshold)

    annotated_bgr = bgr_from_rgb(annotated)
    annotated_bgr = draw_text(
        annotated_bgr,
        "AUTHORIZED" if is_match else "DENIED",
        y=60,
        color=(0, 255, 0) if is_match else (0, 0, 255)
    )
    annotated = rgb_from_bgr(annotated_bgr)

    result_text = (
        f"인증 {'성공' if is_match else '실패'}\n"
        f"- 가장 유사한 사용자: {best_user['user_name']}\n"
        f"- Euclidean distance: {best_dist:.4f}\n"
        f"- Cosine similarity: {best_cos:.4f}\n"
        f"- Distance threshold: {threshold:.4f}\n"
        f"- Cosine threshold: {cosine_threshold:.4f}"
    )
    return annotated, result_text

def delete_user(user_name):
    user_name = (user_name or "").strip()
    if not user_name:
        return "삭제할 사용자 이름을 입력하세요.", safe_get_registered_users_text()

    try:
        deleted_count = delete_user_from_db(user_name)
        if deleted_count > 0:
            return f"삭제 완료: {user_name}", safe_get_registered_users_text()
        return f"삭제 대상 없음: {user_name}", safe_get_registered_users_text()
    except Exception as e:
        return f"삭제 실패: {e}", safe_get_registered_users_text()

def refresh_users():
    return safe_get_registered_users_text()

In [9]:
try:
    init_database()
    db_status_text = "MySQL 연결 및 DB 초기화 성공"
except Exception as e:
    db_status_text = f"MySQL 초기화 실패: {e}"

print(db_status_text)

MySQL 연결 및 DB 초기화 성공


In [10]:
DESCRIPTION = """
# MediaPipe + Gradio + SQLAlchemy(PyMySQL) 얼굴 등록/인증 예제

동작 순서
1. 얼굴 등록 탭에서 이름 입력 후 웹캠 촬영
2. MediaPipe가 얼굴 랜드마크 벡터 추출
3. 벡터를 MySQL DB에 저장
4. 얼굴 인증 탭에서 다시 촬영
5. DB에 저장된 얼굴들과 비교
6. 성공 / 실패 판정

Jupyter에서 마지막 셀을 실행하면 브라우저 링크가 열립니다.
"""

with gr.Blocks(title="MySQL 얼굴 인증", theme=gr.themes.Soft()) as demo:
    gr.Markdown(DESCRIPTION)

    db_status = gr.Textbox(
        label="DB 상태",
        value=db_status_text,
        interactive=False
    )

    user_list_box = gr.Textbox(
        label="등록된 사용자 목록",
        value=safe_get_registered_users_text() if "성공" in db_status_text else "DB 연결 실패 상태",
        lines=8,
        interactive=False
    )

    refresh_btn = gr.Button("사용자 목록 새로고침")

    with gr.Tab("얼굴 등록"):
        with gr.Row():
            enroll_input = gr.Image(
                label="등록용 웹캠 촬영",
                sources=["webcam"],
                type="numpy"
            )
            enroll_output = gr.Image(
                label="등록 결과 이미지",
                type="numpy"
            )

        enroll_name = gr.Textbox(
            label="사용자 이름",
            placeholder="예: haram"
        )
        enroll_btn = gr.Button("얼굴 등록", variant="primary")
        enroll_msg = gr.Textbox(label="등록 결과", lines=3)

    with gr.Tab("얼굴 인증"):
        with gr.Row():
            verify_input = gr.Image(
                label="인증용 웹캠 촬영",
                sources=["webcam"],
                type="numpy"
            )
            verify_output = gr.Image(
                label="인증 결과 이미지",
                type="numpy"
            )

        distance_threshold = gr.Slider(
            minimum=0.5,
            maximum=3.0,
            value=1.8,
            step=0.01,
            label="Distance threshold"
        )

        cosine_threshold = gr.Slider(
            minimum=0.90,
            maximum=1.00,
            value=0.997,
            step=0.0001,
            label="Cosine threshold"
        )

        verify_btn = gr.Button("얼굴 인증", variant="primary")
        verify_msg = gr.Textbox(label="인증 결과", lines=6)

    with gr.Tab("사용자 삭제"):
        delete_name = gr.Textbox(label="삭제할 사용자 이름")
        delete_btn = gr.Button("사용자 삭제", variant="stop")
        delete_msg = gr.Textbox(label="삭제 결과", lines=2)

    enroll_btn.click(
        fn=enroll_face,
        inputs=[enroll_input, enroll_name],
        outputs=[enroll_output, enroll_msg, user_list_box]
    )

    verify_btn.click(
        fn=verify_face,
        inputs=[verify_input, distance_threshold, cosine_threshold],
        outputs=[verify_output, verify_msg]
    )

    delete_btn.click(
        fn=delete_user,
        inputs=[delete_name],
        outputs=[delete_msg, user_list_box]
    )

    refresh_btn.click(
        fn=refresh_users,
        inputs=[],
        outputs=[user_list_box]
    )

IMPORTANT: You are using gradio version 4.29.0, however version 4.44.1 is available, please upgrade.
--------


In [11]:
try:
    demo.close()
except:
    pass

demo.launch(
    server_name="127.0.0.1",
    server_port=7860,
    inline=False,
    inbrowser=True,
    share=False,
    prevent_thread_lock=True,
    show_error=True
)

Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "C:\Users\Admin\miniforge3\envs\meddiapipe\Lib\site-packages\uvicorn\protocols\http\h11_impl.py", line 410, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Admin\miniforge3\envs\meddiapipe\Lib\site-packages\uvicorn\middleware\proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Admin\miniforge3\envs\meddiapipe\Lib\site-packages\fastapi\applications.py", line 1054, in __call__
    await super().__call__(scope, receive, send)
  File "C:\Users\Admin\miniforge3\envs\meddiapipe\Lib\site-packages\starlette\applications.py", line 123, in __call__
    await self.middleware_stack(scope, receive, send)
  File "C:\Users\Admin\miniforge3\envs\meddiapipe\Lib\site-packages\starlette\middleware\errors.py", line 186, in __c

In [13]:
# demo.close()